In [1]:
import pandas as pd

# =============================
# Dataset Loading
# =============================

games_df = pd.read_csv("../Data/api data/Old data/Final Database/games.csv")
detailed_games_df = pd.read_csv("../Data/api data/Old data/Final Database/detailed_games.csv")
company_games_df = pd.read_csv("../Data/api data/Old data/Final Database/company_games.csv")
genre_games_df = pd.read_csv("../Data/api data/Old data/Final Database/genre_games.csv")
platform_df = pd.read_csv("../Data/api data/Old data/Final Database/platform.csv")


final_df = pd.read_csv("../Data/Dataset.csv")

C:\Users\laure\AppData\Local\Temp\ipykernel_16592\1055050788.py:14: DtypeWarning: Columns (8,25,33) have mixed types. Specify dtype option on import or set low_memory=False.
  final_df = pd.read_csv("../Data/Dataset.csv")


In [2]:
final_df.head()
final_df.columns
print(final_df.shape)
print(final_df[['id', 'name']].head())



(159463, 81)
         id                     name
0   58172.0                Spacewar!
1  379437.0           ZeusOfTheCrows
2   56681.0  The Oregon Trail (1971)
3   55012.0                   Empire
4   58174.0           Computer Space


In [22]:
import dash
from dash import dcc, html, Input, Output, State, ctx, ALL
import dash_bootstrap_components as dbc
import threading
import webbrowser
import plotly.express as px
import plotly.graph_objects as go
import socket
import pandas as pd
import ast
import statsmodels.api as sm
import plotly.graph_objects as go
from dash import MATCH

# =========================
# App Setup
# =========================

app = dash.Dash(__name__, external_stylesheets=[dbc.themes.BOOTSTRAP], suppress_callback_exceptions=True)
app.title = "Dashboard"
years = list(range(1970, 2025))

def get_free_port():
    s = socket.socket()
    s.bind(('', 0))
    port = s.getsockname()[1]
    s.close()
    return port

PORT = get_free_port()

# =========================
# Component Builders
# =========================

def build_top_control(active_tab):
    return dbc.ButtonGroup([
        dbc.Button("Games", id="games-button", n_clicks=0, color="primary" if active_tab == "games" else "secondary"),
        dbc.Button("Trends", id="trends-button", n_clicks=0, color="primary" if active_tab == "trends" else "secondary")
    ], id="top_control", style={"width": "100%"})

def build_search_bar():
    return dbc.Card(
        dbc.CardBody([
            dbc.Input(id="search_bar", placeholder="Search...", type="text", debounce=False),
            html.Div(id="search_results", style={
                "position": "absolute", "zIndex": 1000, "width": "100%", "backgroundColor": "white",
                "border": "1px solid #ced4da", "borderRadius": "0.25rem", "boxShadow": "0 2px 6px rgba(0,0,0,0.2)",
                "maxHeight": "200px", "overflowY": "auto", "marginTop": "2px"
            })
        ]),
        style={"position": "relative", "marginTop": "1rem", "marginBottom": "1rem", "backgroundColor": "white"}
    )

def build_games_middle(selected_sub, selected_sort_options):
    options = ["Most Popular"]
    buttons = [
        dbc.Button(
            label,
            id={"type": "sub-button", "index": label.lower().replace(" ", "-") + "-sub"},
            color="primary" if selected_sub == label.lower().replace(" ", "-") + "-sub" else "secondary",
            n_clicks=0,
            style={"width": "100%", "marginBottom": "0.5rem"}
        )
        for label in options
    ]

    sort_options = ["Rating", "YouTube", "Twitch", "Added", "Metacritic"]
    sort_buttons = [
        dbc.Button(
            m,
            id={"type": "sort-button", "index": m},
            color="primary" if m in selected_sort_options else "secondary",
            style={"width": "100%", "marginBottom": "0.25rem"}
        )
        for m in sort_options
    ]

    return html.Div([
        dbc.Row([
            dbc.Col(buttons, width=6),
            dbc.Col([
                html.H6("Sort By:"),
                *sort_buttons
            ], width=6)
        ]),
    ])


def build_sidebar(active_tab, selected_sub, selected_sort_options):
    # === Games tab controls ===
    games_buttons = html.Div(
        build_games_middle(selected_sub, selected_sort_options),
        style={"display": "block" if active_tab == "games" else "none"}
    )
    
    # === Trends tab controls ===
    trends_controls = html.Div([
        html.Label("Show games by:", style={"fontWeight": "bold", "fontSize": "1.1rem", "marginBottom": "0.5rem"}),

        dcc.RadioItems(
            id="trends-sort-radio",
            options=[
                {"label": "Rating", "value": "rating"},
                {"label": "Added", "value": "added"},
                {"label": "Metacritic", "value": "metacritic"},
                {"label": "YouTube", "value": "youtube_count"},
                {"label": "Twitch", "value": "twitch_count"},
            ],
            value="rating",
            labelStyle={
                "display": "block",
                "marginBottom": "0.6rem",
                "fontSize": "1.1rem",  # Increase text size
                "cursor": "pointer"
            },
            style={
                "marginBottom": "1rem",
                "fontSize": "1.1rem",  # This controls the button size (circle)
            }
        ),

        # trendline checkbox
        dbc.Checkbox(
            id="show-trendline-checkbox",
            value=False,
            label="Show trendline",
            style={"marginLeft": "0.5rem"}
        ),
        
        # show average checkbox
        dbc.Checkbox(
            id="show-genre-average-checkbox",
            value=False,
            label="Show averages",
            style={"marginLeft": "0.5rem", "marginTop": "0.5rem"}
        ),

        html.Div([
            html.Label("Select Genre(s)", style={"fontWeight": "bold", "marginTop": "1rem"}),
            dcc.Dropdown(
                id="genre-dropdown",
                options=[],  # Dynamically filled
                multi=True,
                placeholder="Select genres...",
                style={"marginBottom": "1rem"}
            )
        ]),

        html.Div([
            html.Label("Select Year Range", style={"fontWeight": "bold", "marginTop": "1rem"}),
            dcc.RangeSlider(
                id="trends-year-range-slider",
                min=1970,
                max=2024,
                step=1,
                value=[2000, 2020],
                marks={y: str(y) for y in range(1970, 2025, 10)},
                tooltip={"placement": "bottom"}
            ),

            html.Label("Number of Games", style={"fontWeight": "bold", "marginTop": "1.5rem"}),
            dcc.Slider(
                id="trends-num-games-slider",
                min=1000, max=9000, step=500, value=3000,
                marks={i: str(i) for i in range(1000, 9001, 2000)},
                tooltip={"placement": "bottom"}
            )
        ], style={"marginTop": "1rem"})

    ], style={"display": "block" if active_tab == "trends" else "none"})

    # === Sliders ===
    year_slider = html.Div([
        html.Label("Select Release Year Range", style={"fontWeight": "bold", "marginTop": "1rem"}),
        dcc.RangeSlider(
            id="year-range-slider",
            min=1970,
            max=2024,
            step=1,
            marks={y: str(y) for y in range(1970, 2025, 10)},
            value=[2000, 2020],
            tooltip={"placement": "bottom", "always_visible": False},
            allowCross=False,
            updatemode="mouseup"
        )
    ], style={
        "marginTop": "1.5rem", "padding": "1rem",
        "backgroundColor": "#f1f3f5", "borderRadius": "0.5rem",
        "boxShadow": "inset 0 1px 3px rgba(0,0,0,0.1)", "width": "100%",
        "display": "none" if active_tab == "trends" else "block"
    })

    num_slider = html.Div([
        html.Label("Number of games (3 – 200)", style={"fontWeight": "bold", "marginTop": "1rem"}),
        dcc.Slider(
            id="num-games-slider",
            min=3, max=200, step=1, value=20,
            marks={i: str(i) for i in range(10, 201, 30)},
            updatemode="drag"
        )
    ], style={
        "marginTop": "1.5rem", "padding": "1rem",
        "backgroundColor": "#f1f3f5", "borderRadius": "0.5rem",
        "boxShadow": "inset 0 1px 3px rgba(0,0,0,0.1)", "width": "100%",
        "display": "none" if active_tab == "trends" else "block"
    })

    return html.Div([
        build_search_bar(),
        html.Hr(),
        build_top_control(active_tab),
        html.Hr(),
        html.Div(
            id="middle_options",
            children=[games_buttons, trends_controls, year_slider, num_slider],
            style={"paddingTop": "1rem", "paddingBottom": "1rem"}
        )
    ], style={"padding": "1rem"})



def build_data_view():
    return html.Div([
        html.Div(dcc.Loading(html.Div(id="main_graph")), style={"minHeight": "60vh"}),
        html.Hr(),
        html.Div(id="comparison_panel")
    ])


# =========================
# App Layout
# =========================

initial_active_tab = "games"
initial_selected_sub = "most-popular-sub"

app.layout = dbc.Container([
    dcc.Store(id="active_main_tab", data=initial_active_tab),
    dcc.Store(id="selected_sub_button", data=initial_selected_sub),
    dcc.Store(id="selected_sort_options", data=[]),
    dcc.Store(id="selected_year_range", data=[2000, 2020]),
    dcc.Store(id="selected_game_ids", data=[]),
    dcc.Store(id="trends_sort_metric", data="Rating"),
    dcc.Store(id="show_trendline", data=False),
    dcc.Store(id="trends_year_range", data=[2000, 2020]),
    dcc.Store(id="trends_num_games", data=1000),
    dcc.Store(id="last_clicked_timestamp"),

    dbc.Row([
        dbc.Col(id="sidebar", children=build_sidebar(initial_active_tab, initial_selected_sub, []), width=3,
                style={"backgroundColor": "#f8f9fa", "height": "100vh", "padding": 0,
                       "borderRight": "1px solid #dee2e6", "display": "flex", "flexDirection": "column"}),
        dbc.Col(build_data_view(), width=9)
    ])
], fluid=True)

# =========================
# Callbacks
# =========================

@app.callback(
    Output("search_results", "children"),
    Input("search_bar", "value"),
    prevent_initial_call=True
)
def update_search_results(search_text):
    if not search_text or len(search_text.strip()) < 2:
        return ""

    mask = final_df["name"].str.contains(search_text, case=False, na=False)
    matches = final_df.loc[mask].head(10)

    if matches.empty:
        return html.Small("No matches", style={"color": "#888"})

    return dbc.ListGroup([
        dbc.ListGroupItem(
            html.Span(
                row["name"],
                id={"type": "game-link", "index": int(row["id"])},
                n_clicks=0,
                style={"color": "#0d6efd", "cursor": "pointer", "textDecoration": "underline"}
            ),
            style={"padding": "0.4rem 0.6rem"}
        )
        for _, row in matches.iterrows()
    ], flush=True)

@app.callback(
    Output("selected_sort_options", "data"),
    Input({"type": "sort-button", "index": ALL}, "n_clicks"),
    State("selected_sort_options", "data"),
    prevent_initial_call=True
)
def select_sort_option(n_clicks_list, selected_sort_options):
    triggered = ctx.triggered_id
    if not triggered:
        return dash.no_update

    sort_option = triggered["index"]
    return [sort_option]


@app.callback(
    Output("comparison_panel", "children"),
    Input("selected_game_ids", "data"),
    prevent_initial_call=True
)
def update_comparison_panel(game_ids):
    if not game_ids:
        return html.Div("") # panel description effjes leeg

    cards = []
    for game_id in game_ids:
        match = final_df[final_df["id"] == game_id]
        if match.empty:
            continue
        row = match.iloc[0]

        card = dbc.Col([
            dbc.Card([
                dbc.CardHeader(
                    html.Div([
                        html.Span(row.get("name", "Unnamed Game"), style={"fontWeight": "bold", "fontSize": "1.2rem"}),
                        html.Button("×", id={"type": "remove-game", "index": game_id},
                                    style={
                                        "float": "right", "border": "none", "background": "none",
                                        "fontSize": "20px", "cursor": "pointer", "color": "red"
                                    })
                    ])
                ),
                html.Div([
                    html.Div([
                        html.P(f"Rating: {row.get('rating', 'N/A')}"),
                        html.P(f"Metacritic: {row.get('metacritic', 'N/A')}"),
                        html.P(f"Released: {row.get('released', 'Unknown')}"),
                        html.P(f"Added by users: {row.get('added', 'N/A')}"),
                        html.P(f"YouTube count: {row.get('youtube_count', 'N/A')}"),
                        html.P(f"Twitch count: {row.get('twitch_count', 'N/A')}"),
                        html.P(
                            (row.get("description_raw") or "")[:180] + "..."
                        ) if isinstance(row.get("description_raw"), str) else html.P("No description available.")
                    ], style={
                        "position": "absolute",
                        "top": 0,
                        "left": 0,
                        "right": 0,
                        "bottom": 0,
                        "backgroundColor": "rgba(0,0,0,0.55)",
                        "padding": "0.75rem",
                        "borderRadius": "0 0 8px 8px",
                        "color": "white"
                    })
                ], style={
                    "position": "relative",
                    "backgroundImage": f"url('{row.get('background_image', '')}')",
                    "backgroundSize": "cover",
                    "backgroundPosition": "center",
                    "borderRadius": "8px",
                    "height": "240px",
                    "overflow": "hidden"
                })
            ], style={"margin": "0.5rem", "height": "300px", "overflow": "hidden"})
        ], width=6)
        cards.append(card)

    return dbc.Row(cards)




@app.callback(
    Output("selected_game_ids", "data"),
    Output("last_clicked_timestamp", "data"),
    Input({"type": "game-link", "index": ALL}, "n_clicks_timestamp"),
    Input({"type": "remove-game", "index": ALL}, "n_clicks"),
    State({"type": "game-link", "index": ALL}, "id"),
    State("selected_game_ids", "data"),
    State("last_clicked_timestamp", "data"),
    prevent_initial_call=True
)
def handle_game_selection_and_removal(n_clicks_ts, remove_clicks, ids, selected_ids, last_ts):
    if selected_ids is None:
        selected_ids = []

    triggered = ctx.triggered_id

    # Handle removal
    if isinstance(triggered, dict) and triggered.get("type") == "remove-game":
        removed_id = triggered["index"]
        return [gid for gid in selected_ids if gid != removed_id], last_ts

    # Handle selection
    max_ts = -1
    clicked_id = None

    for ts, btn_id in zip(n_clicks_ts, ids):
        if ts is not None and ts > max_ts:
            max_ts = ts
            clicked_id = btn_id["index"]

    if clicked_id is not None and (last_ts is None or max_ts > last_ts):
        if clicked_id in selected_ids:
            return selected_ids, max_ts

        if len(selected_ids) == 2:
            selected_ids.pop(0)

        selected_ids.append(clicked_id)
        return selected_ids, max_ts

    return dash.no_update, dash.no_update

@app.callback(
    Output("active_main_tab", "data"),
    Input("games-button", "n_clicks"),
    Input("trends-button", "n_clicks"),
    prevent_initial_call=True
)
def update_active_tab(games_clicks, trends_clicks):
    triggered_id = ctx.triggered_id
    if triggered_id == "games-button":
        return "games"
    elif triggered_id == "trends-button":
        return "trends"
    return dash.no_update


@app.callback(
    Output("sidebar", "children"),
    Input("active_main_tab", "data"),
    State("selected_sub_button", "data"),
    State("selected_sort_options", "data"),
)
def update_sidebar(active_tab, selected_sub, selected_sort_options):
    return build_sidebar(active_tab, selected_sub, selected_sort_options)

@app.callback(
    Output("trends_sort_metric", "data"),
    Input("trends-sort-radio", "value")
)
def update_sort_metric(value):
    return value

@app.callback(
    Output("show_trendline", "data"),
    Input("show-trendline-checkbox", "value"),
    prevent_initial_call=True
)
def update_trendline_state(show):
    return show or False

@app.callback(Output("trends_year_range", "data"), Input("trends-year-range-slider", "value"))
def update_trends_year_range(value): return value

@app.callback(Output("trends_num_games", "data"), Input("trends-num-games-slider", "value"))
def update_trends_num_games(value): return value

@app.callback(
    Output("genre-dropdown", "options"),
    Input("active_main_tab", "data")
)
def populate_genre_options(tab):
    if tab != "trends":
        return []

    genre_set = set()
    for g in final_df["genres"].dropna():
        for genre in g.split(","):
            genre_set.add(genre.strip())

    return [{"label": genre, "value": genre} for genre in sorted(genre_set)]



# =========================
# MAIN GRAPH – HEAT-MAP VERSION
# =========================

import plotly.express as px
import squarify
import numpy as np

@app.callback(
    Output("main_graph", "children"),
    Input("active_main_tab", "data"),
    Input("selected_sort_options", "data"),

    # Heatmap
    Input("selected_year_range", "data"),
    Input("num-games-slider", "value"),

    # Trends
    Input("trends_year_range", "data"),
    Input("trends_num_games", "data"),
    Input("genre-dropdown", "value"),
    Input("trends_sort_metric", "data"),
    Input("show_trendline", "data"),
    Input("show-genre-average-checkbox", "value")
)


def update_main_graph(
    active_tab,
    selected_sort_options,
    heatmap_year_range,
    heatmap_num_games,
    trends_year_range,
    trends_num_games,
    selected_genres,
    trends_sort_metric,
    show_trendline,
    show_genre_average):
    if active_tab == "trends":
        selected_genres = selected_genres or []
        year_range = trends_year_range
        num_games = trends_num_games
    else:
        year_range = heatmap_year_range
        num_games = heatmap_num_games

    df = final_df.copy()
    df = df.dropna(subset=["name"]).drop_duplicates()
    df["release_year"] = pd.to_datetime(df["released"], errors="coerce").dt.year
    df = df[
        (df["release_year"] >= year_range[0]) &
        (df["release_year"] <= year_range[1])
    ]


    # == Show Trends: Scatterplot ==
    if active_tab == "trends":

        def extract_matching_genre(genre_str):
            try:
                genre_list = [g.strip() for g in genre_str.split(",")]
                for g in genre_list:
                    if g in selected_genres:
                        return g
            except:
                pass
            return None


        filtered_df = df.dropna(subset=["released", "rating", "genres"])
        filtered_df["released"] = pd.to_datetime(filtered_df["released"], errors="coerce")
        filtered_df = filtered_df[filtered_df["rating"] > 0.5]
        filtered_df = filtered_df[
            (filtered_df["released"].dt.year >= year_range[0]) &
            (filtered_df["released"].dt.year <= year_range[1])
        ]

        # Match genre
        if selected_genres:
            filtered_df["matched_genre"] = filtered_df["genres"].apply(extract_matching_genre)
            filtered_df = filtered_df[filtered_df["matched_genre"].notna()]
            color_column = "matched_genre"
        else:
            filtered_df["matched_genre"] = "All"
            color_column = None

        # Sample and sort
        filtered_df = filtered_df.sample(n=min(num_games, len(filtered_df)), random_state=42)
        filtered_df = filtered_df.sort_values("released")

        # Clean Y-axis metric
        y_metric = "rating" if trends_sort_metric.lower() == "rating" else "added"
        filtered_df = filtered_df.dropna(subset=["released", y_metric])
        filtered_df[y_metric] = pd.to_numeric(filtered_df[y_metric], errors="coerce")
        filtered_df = filtered_df.dropna(subset=[y_metric])

        x = filtered_df["released"].map(pd.Timestamp.toordinal)
        y = filtered_df[y_metric]
        yaxis_label = {
            "rating": "Rating",
            "added": "Added by Users",
            "metacritic": "Metacritic Score",
            "youtube_count": "YouTube Mentions",
            "twitch_count": "Twitch Mentions"
        }.get(y_metric, y_metric.capitalize())


        fig = go.Figure()
        
        if show_genre_average:
            # Group by year and genre, compute average of y_metric
            filtered_df["year"] = filtered_df["released"].dt.year
            grouped = (
                filtered_df.groupby(["year", "matched_genre"])[y_metric]
                .mean()
                .reset_index()
            )

            for genre in grouped["matched_genre"].unique():
                genre_df = grouped[grouped["matched_genre"] == genre]
                fig.add_trace(go.Scatter(
                    x=genre_df["year"],
                    y=genre_df[y_metric],
                    mode="lines+markers",
                    name=genre,
                    marker=dict(size=6, opacity=0.8),
                    line=dict(width=2)
                ))

        else:
            # Plot individual games
            for genre in filtered_df["matched_genre"].unique():
                genre_df = filtered_df[filtered_df["matched_genre"] == genre]
                fig.add_trace(go.Scatter(
                    x=genre_df["released"],
                    y=genre_df[y_metric],
                    mode="markers",
                    name=genre,
                    text=genre_df["name"],
                    marker=dict(size=6, opacity=0.6)
                ))


        # Trendline
        if show_trendline and not filtered_df.empty and len(x) > 1:
            X = sm.add_constant(x)
            model = sm.OLS(y, X).fit()
            trend_y = model.predict(X)

            fig.add_trace(go.Scatter(
                x=filtered_df["released"],
                y=trend_y,
                mode="lines",
                name="Trendline",
                line=dict(color="orange", width=2)
            ))

        # Auto-set Y-axis range based on data
        y_min = y.min()
        y_max = y.max()
        padding = (y_max - y_min) * 0.05 if y_max > y_min else 1
        yaxis_range = [max(0, y_min - padding), y_max + padding]

        fig.update_layout(
            title=f"Game {yaxis_label} Over Time" + (" by Genre" if selected_genres else ""),
            xaxis_title="Release Date",
            yaxis_title=yaxis_label,
            yaxis_range=yaxis_range,
            transition_duration=500
        )

        return dcc.Graph(figure=fig)

    # == Otherwise, default to HEATMAP ==
    column_mapping = {
        "rating": "rating",
        "youtube": "youtube_count",
        "twitch": "twitch_count",
        "added": "added",
        "metacritic": "metacritic",
    }
    sort_by = column_mapping.get(
        (selected_sort_options[0] if selected_sort_options else "rating").lower(),
        "rating",
    )

    num_games = num_games or 50           # fallback default
    df = (
        df.dropna(subset=[sort_by])
          .sort_values(sort_by, ascending=False)
          .head(num_games)
    )

    # 2. Treemap areas – **linear** with the metric
    #    Guard against zeros so we don't hand 0-area to squarify.
    metric = df[sort_by].astype(float).clip(lower=1e-6)
    areas  = metric                       # <- direct proportionality
    normed = squarify.normalize_sizes(areas, 100, 100)
    rects  = squarify.squarify(normed, 0, 0, 100, 100)
    df = pd.concat([df.reset_index(drop=True), pd.DataFrame(rects)], axis=1)

    # 3. Build tiles (unchanged visual style)
    tiles = [
        html.Div(
            children=[
                html.Div(
                    [
                        html.Div(
                            row["name"],
                            style={
                                "fontSize": "12px", "fontWeight": "bold",
                                "overflow": "hidden", "textOverflow": "ellipsis",
                                "whiteSpace": "nowrap",
                            },
                        ),
                        html.Div(
                            f"{sort_by.capitalize()}: {row[sort_by]:.2f}",
                            style={"fontSize": "10px"},
                        ),
                    ],
                    style={
                        "position": "absolute", "inset": 0,
                        "backgroundColor": "rgba(0,0,0,0.55)",
                        "display": "flex", "flexDirection": "column",
                        "alignItems": "center", "justifyContent": "center",
                        "padding": "4px",
                    },
                )
            ],
            style={
                "position": "absolute",
                "left":   f"{row['x']:.2f}%",  "top":    f"{row['y']:.2f}%",
                "width":  f"{row['dx']:.2f}%", "height": f"{row['dy']:.2f}%",
                "backgroundImage":  f"url('{row['background_image']}')",
                "backgroundSize":   "cover",
                "backgroundPosition": "center",
                "border": "1px solid #fff",
                "boxSizing": "border-box",
                "borderRadius": "4px",
                "overflow": "hidden",
                "color": "#fff",
            },
        )
        for _, row in df.iterrows()
    ]

    # 4. Return wrapped container
    return html.Div(
        tiles,
        style={
            "position": "relative",
            "width": "100%",
            "height": "60vh",
            "backgroundColor": "#333",
        },
    )


@app.callback(
    Output("selected_year_range", "data"),
    Input("year-range-slider", "value"),
    prevent_initial_call=True
)
def update_selected_year_range(year_range):
    return year_range


# =========================
# Run server
# =========================

def open_browser():
    webbrowser.open_new(f"http://127.0.0.1:{PORT}")

if __name__ == "__main__":
    threading.Timer(1, open_browser).start()
    app.run(debug=True, use_reloader=False, port=PORT)


C:\Users\laure\AppData\Local\Temp\ipykernel_16592\1663704842.py:547: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

C:\Users\laure\AppData\Local\Temp\ipykernel_16592\1663704842.py:547: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

